In [ ]:
import os
import pandas as pd
from collections import defaultdict
import pickle
import csv

In [ ]:
def group_files(input_dir):
    """
    Groups TSV files based on filename:
    - If the filename contains "KapK", group them under "KapK".
    - Otherwise, group by everything before the first underscore.
    """
    groups = defaultdict(list)
    for filename in os.listdir(input_dir):
        if filename.endswith(".tsv"):
            if "KapK" in filename:
                groups["KapK"].append(os.path.join(input_dir, filename))
            else:
                group_key = filename[0:3]
                groups[group_key].append(os.path.join(input_dir, filename))
    return groups

def merge_tsvs_for_group(file_list):
    """
    Reads and merges TSV files from file_list.
    Assumes each file has a header row.
    """
    df_list = []
    for file_path in file_list:
        try:
            df = pd.read_csv(file_path, sep="\t", header=0)
            df_list.append(df)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    if df_list:
        merged_df = pd.concat(df_list, ignore_index=True)
    else:
        merged_df = pd.DataFrame()
    return merged_df

def filter_lowest_evalue(merged_df):
    """
    For each assembler and target combination, keeps only the hit with the lowest E-value.
    """
    print("Filter Lowest EValue... Start")
    merged_df['evalue'] = pd.to_numeric(merged_df['evalue'], errors='coerce')
    idx = merged_df.groupby(['assembler', 'target'])['evalue'].idxmin()
    filtered_df = merged_df.loc[idx].reset_index(drop=True)
    print("Filter Lowest EValue... Done")
    return filtered_df

def process_directory(input_dir, output_dir, ancient_sample_names, onlyAncient=False):
    """
    Processes all TSV files in input_dir:
    1. Groups files by sample based on filename.
    2. Merges TSVs within each group.
    3. For each assembler, retains only the best (lowest E-value) hit per target.
    4. Saves the merged, filtered results per group.
    """
    groups = group_files(input_dir)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Found {len(groups)} groups: {list(groups.keys())}")
    
    for group_key, file_list in groups.items():
        print(f"Processing group '{group_key}' with {len(file_list)} file(s).")
        merged_df = merge_tsvs_for_group(file_list)
        if merged_df.empty:
            print(f"No data for group '{group_key}'.")
            continue
        filtered_df = filter_lowest_evalue(merged_df)
        filtered_df = filtered_df[filtered_df["query"].isin(ancient_sample_names)]
        
        output_path = os.path.join(output_dir, f"{group_key}_merged.tsv")
        filtered_df.to_csv(output_path, sep="\t", index=False)
        print(f"Saved merged file for group '{group_key}' with {len(filtered_df)} rows to {output_path}")



#### Only ancient

def load_ancient_ids(ancient_ids_file):
    """
    Load the ancient IDs file and return a list of SampleName values.
    The file is assumed to be a TSV with a header including "SampleName".
    """
    sample_names = []
    with open(ancient_ids_file, 'r', newline='') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            sample_names.append(row['SampleName'].rsplit('_', 1)[0])
    return sample_names


In [ ]:
dir_in="/project3/data/results/assembly-prokka-mmseqs-nr/merged"
dir_out="/project3/data/results/assembly-prokka-mmseqs-nr/merged/groupedAncientOnly"
ancient_ids_file = "/project3/data/results/other-evals/ancientness/ancient_orfs_min100.tsv"  # Path to the ancient IDs file (TSV)
onlyAncient = True

In [ ]:
# Load ancient sample names
ancient_sample_names = ""
if onlyAncient:
    ancient_sample_names = load_ancient_ids(ancient_ids_file)

In [ ]:
#ancient_sample_names

In [ ]:
process_directory(dir_in, dir_out, ancient_sample_names, onlyAncient)

In [ ]:
### remove duplicate queries

# Read the TSV file
df = pd.read_csv("/project3/data/results/assembly-prokka-mmseqs-nr/merged/groupedAncientOnly/KapK_merged.tsv", sep="\t")

# Convert the evalue column to numeric (in case it's read as string)
df['evalue'] = pd.to_numeric(df['evalue'], errors='coerce')

# Sort by evalue (ascending: best hits first)
df_sorted = df.sort_values(by='evalue', ascending=True)

# Drop duplicate queries, keeping the first (best) one
df_best_hits = df_sorted.drop_duplicates(subset='query', keep='first')

# Save to file
df_best_hits.to_csv("/project3/data/results/assembly-prokka-mmseqs-nr/merged/groupedAncientOnly/KapK_merged_besthits.tsv", sep="\t", index=False)
